# T1CIR Tutorial: Building Neural Network Graphs

This tutorial introduces T1CIR (Type-1 Compute Intermediate Representation), a hardware-agnostic format for representing spiking and hybrid neural networks as directed graphs.

## What You'll Learn

1. What T1CIR is and why it exists
2. Creating nodes (primitives) for your network
3. Building graphs by connecting nodes
4. Serializing graphs to disk (HDF5 format)
5. Graph validation and inspection
6. Building convolutional networks
7. Adding skip connections

## Prerequisites

```bash
pip install t1cir
```

## 1. Introduction to T1CIR

T1CIR is designed to:

- **Represent** spiking neural networks as computational graphs
- **Bridge** PyTorch/SNNTorch models to hardware deployment
- **Support** both SNN and hybrid ANN-SNN architectures
- **Enable** hardware-agnostic model exchange

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Graph** | Container for nodes and edges forming a DAG |
| **Node** | A computational primitive (LIF, Conv2d, etc.) |
| **Edge** | Connection between nodes (data flow) |
| **Primitive** | The type of operation a node performs |

In [1]:
# Setup
import numpy as np
from t1c import ir

print(f"T1C-IR version: {ir.__version__}")
print(f"\nAvailable primitives: {len(ir.list_primitives())}")
for p in sorted(ir.list_primitives()):
    print(f"  - {p}")

T1C-IR version: 0.0.1

Available primitives: 22
  - Affine
  - AvgPool2d
  - BatchNorm1d
  - BatchNorm2d
  - Conv2d
  - Dropout
  - ELU
  - Flatten
  - GELU
  - HybridRegion
  - LIF
  - LayerNorm
  - MaxPool2d
  - PReLU
  - ReLU
  - SepConv2d
  - Sigmoid
  - Skip
  - Softmax
  - SpikingAffine
  - Tanh
  - Upsample


## 2. Creating Nodes (Primitives)

Every neural network is made of computational operations. In T1CIR, these are called **primitives**.

### Essential Primitives

| Primitive | Purpose | Parameters |
|-----------|---------|------------|
| `Input` | Entry point | shape |
| `Output` | Exit point | shape |
| `Affine` | Fully-connected layer | weight, bias |
| `LIF` | Leaky Integrate-and-Fire neuron | tau, r, v_leak, v_threshold |
| `Conv2d` | 2D convolution | weight, bias, stride, padding |

In [2]:
# Create an Input node
# This defines the shape of data entering the network
input_node = ir.Input(np.array([784]))  # Flattened 28x28 image
print(f"Input shape: {input_node.input_type}")

Input shape: {'input': array([784])}


In [3]:
# Create an Affine (fully-connected) layer
# weight shape: (out_features, in_features)
fc1 = ir.Affine(
    weight=np.random.randn(128, 784).astype(np.float32) * 0.01,
    bias=np.zeros(128, dtype=np.float32),
)
print(f"Affine weight shape: {fc1.weight.shape}")
print(f"Affine bias shape: {fc1.bias.shape}")

Affine weight shape: (128, 784)
Affine bias shape: (128,)


In [4]:
# Create a LIF (Leaky Integrate-and-Fire) neuron
# LIF neurons are the core of spiking neural networks
#
# Parameters:
#   tau: membrane time constant (higher = slower decay)
#   r: membrane resistance
#   v_leak: resting membrane potential
#   v_threshold: spike threshold

lif1 = ir.LIF(
    tau=np.ones(128, dtype=np.float32) * 10.0,      # 10ms time constant
    r=np.ones(128, dtype=np.float32),               # unit resistance
    v_leak=np.zeros(128, dtype=np.float32),         # 0V resting potential
    v_threshold=np.ones(128, dtype=np.float32),     # 1V threshold
)
print(f"LIF neurons: {len(lif1.tau)}")

LIF neurons: 128


In [5]:
# Create the output layer
fc2 = ir.Affine(
    weight=np.random.randn(10, 128).astype(np.float32) * 0.01,
    bias=np.zeros(10, dtype=np.float32),
)

lif2 = ir.LIF(
    tau=np.ones(10, dtype=np.float32) * 10.0,
    r=np.ones(10, dtype=np.float32),
    v_leak=np.zeros(10, dtype=np.float32),
    v_threshold=np.ones(10, dtype=np.float32),
)

output_node = ir.Output(np.array([10]))
print(f"Output shape: {output_node.output_type}")

Output shape: {'output': array([10])}


## 3. Building a Graph

A graph is created by:
1. Defining a dictionary of named nodes
2. Defining a list of edges (source, destination) tuples

```
Input -> Affine -> LIF -> Affine -> LIF -> Output
```

In [6]:
# Define nodes with names
nodes = {
    "input": input_node,
    "fc1": fc1,
    "lif1": lif1,
    "fc2": fc2,
    "lif2": lif2,
    "output": output_node,
}

# Define edges (data flow)
edges = [
    ("input", "fc1"),
    ("fc1", "lif1"),
    ("lif1", "fc2"),
    ("fc2", "lif2"),
    ("lif2", "output"),
]

# Create the graph
graph = ir.Graph(nodes=nodes, edges=edges)

print(f"Graph created:")
print(f"  Nodes: {len(graph.nodes)}")
print(f"  Edges: {len(graph.edges)}")
print(f"  Is DAG: {graph.is_dag}")

Graph created:
  Nodes: 6
  Edges: 5
  Is DAG: True


In [7]:
# Inspect the graph structure
print("Nodes:")
for name, node in graph.nodes.items():
    print(f"  {name}: {type(node).__name__}")

print("\nEdges:")
for src, dst in graph.edges:
    print(f"  {src} -> {dst}")

Nodes:
  input: Input
  fc1: Affine
  lif1: LIF
  fc2: Affine
  lif2: LIF
  output: Output

Edges:
  input -> fc1
  fc1 -> lif1
  lif1 -> fc2
  fc2 -> lif2
  lif2 -> output


## 4. Serialization (Save/Load)

T1CIR uses HDF5 format for serialization. This preserves:
- Graph structure (nodes and edges)
- All parameters (weights, biases, etc.)
- Metadata

In [8]:
# Save the graph
import os
os.makedirs("models", exist_ok=True)

output_path = "models/tutorial_simple_snn.t1c"
ir.write(output_path, graph)
print(f"Saved to: {output_path}")

Saved to: models/tutorial_simple_snn.t1c


In [9]:
# Load the graph back
loaded_graph = ir.read(output_path)

print(f"Loaded graph:")
print(f"  Nodes: {len(loaded_graph.nodes)}")
print(f"  Edges: {len(loaded_graph.edges)}")

# Verify weights are preserved
original_weights = fc1.weight
loaded_weights = loaded_graph.nodes["fc1"].weight
weight_diff = np.abs(original_weights - loaded_weights).max()
print(f"  Weight preservation: max diff = {weight_diff:.2e}")

Loaded graph:
  Nodes: 6
  Edges: 5
  Weight preservation: max diff = 0.00e+00


## 5. Graph Validation

T1CIR automatically validates graphs:
- DAG check (no cycles by default)
- Node existence in edges
- Type validation

In [10]:
# Check graph properties
print(f"Is DAG: {graph.is_dag}")
print(f"Node count: {len(graph.nodes)}")
print(f"Edge count: {len(graph.edges)}")

Is DAG: True
Node count: 6
Edge count: 5


In [11]:
# Try creating an invalid graph (cycle)
try:
    cyclic_edges = edges + [("lif2", "fc1")]  # Creates a cycle
    cyclic_graph = ir.Graph(nodes=nodes, edges=cyclic_edges)
    print("Graph created (allows cycles)")
    print(f"  Is DAG: {cyclic_graph.is_dag}")
except ValueError as e:
    print(f"Error: {e}")

Graph created (allows cycles)
  Is DAG: False


## 6. Building a Convolutional Network

For image processing, we use:
- `Conv2d`: 2D convolution
- `MaxPool2d`: Max pooling
- `Flatten`: Reshape for dense layers

```
Input (1x28x28) -> Conv2d (8x28x28) -> LIF -> MaxPool (8x14x14) ->
Conv2d (16x14x14) -> LIF -> MaxPool (16x7x7) -> Flatten -> FC -> Output
```

In [12]:
# Helper function to create conv blocks
def create_conv_block(name, in_channels, out_channels, kernel_size=3):
    """Create a Conv2d + LIF block."""
    conv = ir.Conv2d(
        weight=np.random.randn(out_channels, in_channels, kernel_size, kernel_size).astype(np.float32) * 0.1,
        bias=np.zeros(out_channels, dtype=np.float32),
        stride=(1, 1),
        padding=(kernel_size // 2, kernel_size // 2),
    )
    lif = ir.LIF(
        tau=np.ones(out_channels, dtype=np.float32) * 10.0,
        r=np.ones(out_channels, dtype=np.float32),
        v_leak=np.zeros(out_channels, dtype=np.float32),
        v_threshold=np.ones(out_channels, dtype=np.float32),
    )
    return {f"{name}_conv": conv, f"{name}_lif": lif}

# Build CNN
cnn_nodes = {}
cnn_nodes["input"] = ir.Input(np.array([1, 28, 28]))  # CHW format

# Conv block 1: 1 -> 8 channels
cnn_nodes.update(create_conv_block("block1", 1, 8))
cnn_nodes["pool1"] = ir.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

# Conv block 2: 8 -> 16 channels
cnn_nodes.update(create_conv_block("block2", 8, 16))
cnn_nodes["pool2"] = ir.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))

# Classifier
cnn_nodes["flatten"] = ir.Flatten(start_dim=0)
cnn_nodes["fc"] = ir.Affine(
    weight=np.random.randn(10, 16 * 7 * 7).astype(np.float32) * 0.01,
    bias=np.zeros(10, dtype=np.float32),
)
cnn_nodes["fc_lif"] = ir.LIF(
    tau=np.ones(10, dtype=np.float32) * 10.0,
    r=np.ones(10, dtype=np.float32),
    v_leak=np.zeros(10, dtype=np.float32),
    v_threshold=np.ones(10, dtype=np.float32),
)
cnn_nodes["output"] = ir.Output(np.array([10]))

print(f"CNN nodes: {len(cnn_nodes)}")
for name, node in cnn_nodes.items():
    print(f"  {name}: {type(node).__name__}")

CNN nodes: 11
  input: Input
  block1_conv: Conv2d
  block1_lif: LIF
  pool1: MaxPool2d
  block2_conv: Conv2d
  block2_lif: LIF
  pool2: MaxPool2d
  flatten: Flatten
  fc: Affine
  fc_lif: LIF
  output: Output


In [13]:
# Define CNN edges
cnn_edges = [
    ("input", "block1_conv"),
    ("block1_conv", "block1_lif"),
    ("block1_lif", "pool1"),
    ("pool1", "block2_conv"),
    ("block2_conv", "block2_lif"),
    ("block2_lif", "pool2"),
    ("pool2", "flatten"),
    ("flatten", "fc"),
    ("fc", "fc_lif"),
    ("fc_lif", "output"),
]

cnn_graph = ir.Graph(nodes=cnn_nodes, edges=cnn_edges)
print(f"\nCNN Graph:")
print(f"  Nodes: {len(cnn_graph.nodes)}")
print(f"  Edges: {len(cnn_graph.edges)}")
print(f"  Is DAG: {cnn_graph.is_dag}")


CNN Graph:
  Nodes: 11
  Edges: 10
  Is DAG: True


## 7. Skip Connections (Residual Blocks)

Skip connections allow gradients to flow through the network more easily.

```
      ┌─────────────────────────┐
      │                         │
      │                         ▼
Input -> Conv -> LIF -> Conv -> Skip -> LIF -> Output
```

In [14]:
# Create a residual block
channels = 32

res_nodes = {
    "input": ir.Input(np.array([channels, 14, 14])),
    "conv1": ir.Conv2d(
        weight=np.random.randn(channels, channels, 3, 3).astype(np.float32) * 0.1,
        bias=np.zeros(channels, dtype=np.float32),
        stride=(1, 1),
        padding=(1, 1),
    ),
    "lif1": ir.LIF(
        tau=np.ones(channels, dtype=np.float32) * 10.0,
        r=np.ones(channels, dtype=np.float32),
        v_leak=np.zeros(channels, dtype=np.float32),
        v_threshold=np.ones(channels, dtype=np.float32),
    ),
    "conv2": ir.Conv2d(
        weight=np.random.randn(channels, channels, 3, 3).astype(np.float32) * 0.1,
        bias=np.zeros(channels, dtype=np.float32),
        stride=(1, 1),
        padding=(1, 1),
    ),
    "skip": ir.Skip(
        input_type={'input': np.array([channels, 14, 14])},
        skip_type="residual",
    ),
    "lif2": ir.LIF(
        tau=np.ones(channels, dtype=np.float32) * 10.0,
        r=np.ones(channels, dtype=np.float32),
        v_leak=np.zeros(channels, dtype=np.float32),
        v_threshold=np.ones(channels, dtype=np.float32),
    ),
    "output": ir.Output(np.array([channels, 14, 14])),
}

res_edges = [
    ("input", "conv1"),
    ("conv1", "lif1"),
    ("lif1", "conv2"),
    ("conv2", "skip"),
    ("input", "skip"),      # Skip connection from input
    ("skip", "lif2"),
    ("lif2", "output"),
]

res_graph = ir.Graph(nodes=res_nodes, edges=res_edges)
print(f"Residual Block:")
print(f"  Nodes: {len(res_graph.nodes)}")
print(f"  Edges: {len(res_graph.edges)}")
print(f"  Is DAG: {res_graph.is_dag}")

Residual Block:
  Nodes: 7
  Edges: 7
  Is DAG: True


In [15]:
# Save the residual block
ir.write("models/tutorial_residual.t1c", res_graph)
print("Saved residual block to models/tutorial_residual.t1c")

Saved residual block to models/tutorial_residual.t1c


## Summary

You've learned:

1. **Primitives**: Input, Output, Affine, LIF, Conv2d, MaxPool2d, Skip
2. **Graph construction**: nodes dict + edges list
3. **Serialization**: `t1cir.write()` and `t1cir.read()`
4. **Validation**: `graph.is_dag`, `graph.validate()`
5. **Architectures**: Dense, CNN, Residual

## Next Steps

- **T1CViz Tutorial**: Visualize graphs and spike events
- **T1CTorch Tutorial**: Convert PyTorch models to T1CIR
- **T1C-SDK Tutorial**: Analyze, profile, and deploy graphs